# 01 — Exploratory Data Analysis: CIC-IDS2017

This notebook explores the CIC-IDS2017 MachineLearningCSV dataset:
- Class distribution (BENIGN vs attack types)
- Missing values and infinities
- Feature correlations
- Basic statistics

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)

RAW_DIR = os.path.join('..', 'data', 'raw')
print('CSV files:', os.listdir(RAW_DIR))

## 1. Load & Concatenate

In [ ]:
csv_files = sorted(glob.glob(os.path.join(RAW_DIR, '*.csv')))
frames = []
for f in csv_files:
    df = pd.read_csv(f, low_memory=False, encoding='utf-8-sig')
    df.columns = df.columns.str.strip()
    frames.append(df)
    print(f'{os.path.basename(f):50s}  →  {len(df):>10,} rows')

data = pd.concat(frames, ignore_index=True)
print(f'\nTotal: {len(data):,} rows  ×  {data.shape[1]} columns')

## 2. Class Distribution

In [ ]:
label_col = 'Label'
class_counts = data[label_col].value_counts()
print(class_counts)
print(f'\nNumber of unique classes: {data[label_col].nunique()}')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart
class_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('viridis', len(class_counts)))
axes[0].set_title('Class Distribution (absolute counts)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Log-scale to see minority classes
class_counts.plot(kind='bar', ax=axes[1], color=sns.color_palette('viridis', len(class_counts)))
axes[1].set_yscale('log')
axes[1].set_title('Class Distribution (log scale)')
axes[1].set_ylabel('Count (log)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Percentage
pct = (class_counts / len(data) * 100).round(2)
print('\nClass percentages:')
print(pct)

## 3. Missing Values & Infinities

In [ ]:
# Check NaN
nan_counts = data.isnull().sum()
nan_cols = nan_counts[nan_counts > 0]
print('Columns with NaN values:')
print(nan_cols if len(nan_cols) > 0 else 'None')

# Check infinities
numeric_cols = data.select_dtypes(include=[np.number]).columns
inf_counts = pd.Series(0, index=numeric_cols)
for col in numeric_cols:
    inf_counts[col] = np.isinf(data[col].values.astype(float)).sum()

inf_cols = inf_counts[inf_counts > 0]
print(f'\nColumns with infinity values ({len(inf_cols)}):')
print(inf_cols if len(inf_cols) > 0 else 'None')

total_problematic = nan_counts.sum() + inf_counts.sum()
print(f'\nTotal problematic cells: {total_problematic:,} '
      f'({total_problematic / data.size * 100:.4f}% of all cells)')

## 4. Feature Correlation (Top 20 by Variance)

In [ ]:
# Drop non-numeric and problematic columns for correlation
numeric_data = data.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan).dropna()

# Select top 20 features by variance
variances = numeric_data.var().sort_values(ascending=False)
top_20 = variances.head(20).index.tolist()

corr = numeric_data[top_20].corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Matrix — Top 20 Features by Variance')
plt.tight_layout()
plt.show()

## 5. Basic Statistics

In [ ]:
print(f'Shape: {data.shape}')
print(f'Dtypes:\n{data.dtypes.value_counts()}')
print(f'\nDescriptive statistics (first 10 numeric features):')
data[numeric_cols[:10]].describe().round(2)